# Step 4: Add Metadata and Analyze Spatial Patterns


This notebook turns metric results into spatial summaries. You will use PGA and FAS as two separate examples, so you can see how the same spatial tests can highlight different model-performance patterns for different metrics.


## Imports

Purpose: import package-level spatial workflow, map, summary, and provenance helpers.

Outputs: imports only; no files are written.

These functions prepare metric fields and calculate spatial summaries.


In [ ]:
from pathlib import Path
import runpy

# Make the local source checkout importable when running notebooks without an installed wheel.
_bootstrap = next(
    (
        path
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for path in (
            candidate / "_source_bootstrap.py",
            candidate / "docs" / "examples" / "_source_bootstrap.py",
        )
        if path.exists()
    ),
    None,
)
if _bootstrap is None:
    raise RuntimeError("Could not find docs/examples/_source_bootstrap.py.")
repo_root = runpy.run_path(str(_bootstrap))["use_source_checkout"]()

from spatial_vtk.config import (
    notebook_timer,
    notebook_figure_settings,
    prepare_notebook_geospatial_environment,
    register_svtk_cell_timer,
)
prepare_notebook_geospatial_environment(loky_max_cpu_count=1)

with notebook_timer():

    from IPython.display import Markdown, display

    from spatial_vtk.spatial import (
        load_standard_spatial_workflow_output_status,
        load_standard_spatial_workflow_outputs,
        spatial_workflow_failure_frame,
    )
    register_svtk_cell_timer()


## Configuration

Purpose: load the tutorial config and spatial-statistics settings.

Outputs: an active config plus Step 4 spatial workflow settings.

Load the config and read the spatial-statistics settings from the tutorial run scenario.


In [ ]:
from spatial_vtk.config import notebook_figure_settings, notebook_run_context

config_path = repo_root / "data/examples/configuration/example_spatial_vtk_config.yaml"

# Load the tutorial run scenario and make it the active config for later package calls.
context = notebook_run_context(config_path, run_scenario="tutorial")
cfg = context.cfg
# Step 4 uses the configured larger QC-passed metric snapshot so per-metric spatial tests have enough stations.
spatial_figure_settings = notebook_figure_settings("spatial")


## Prepare Metric-Specific Spatial Fields

Purpose: build metric-specific spatial tables from the Step 3 outputs.

Outputs: metric field, event-centered residual, station-bias, and related spatial summary tables.

Spatial statistics use one value per event-station observation, plus station and event coordinates. This notebook delegates the table-building work to the package workflow, then uses the returned tables for compact displays and figures.


In [ ]:
# Run the config-backed spatial summary workflow when outputs are stale.
# The package helper owns output readiness, table-name mapping, and per-metric summaries.
spatial_status = load_standard_spatial_workflow_output_status(cfg=cfg)
display(spatial_status.status_frame())

spatial_result = spatial_status.run_summary_step_if_needed(
    context,
    overwrite=context.overwrite,
    verbose=True,
    run_local=True,
)

failure_table = spatial_workflow_failure_frame(spatial_result)
if not failure_table.empty:
    display(Markdown("### Non-fatal workflow diagnostics"))
    display(failure_table)

spatial_outputs = load_standard_spatial_workflow_outputs(spatial_result, cfg=cfg)
display(spatial_outputs.status_frame())
display(spatial_outputs.summary_frame())
display(spatial_outputs.station_bias_preview_frame())


## Station Bias Maps

Purpose: render station maps of mean event-centered residuals.

Outputs: station-bias map figures and optional source-row sidecars.

These maps show the mean event-centered residual at each station. Positive values mean the observed amplitudes are larger than the synthetic amplitudes on average for that metric.


In [ ]:
spatial_map_result = spatial_outputs.write_map_figures(spatial_figure_settings)
display(spatial_map_result.status_frame())


## Residual Grid Maps

Purpose: render gridded residual maps for the selected metrics.

Outputs: residual-grid map figures and optional source-row sidecars.

A residual grid gives you a quick spatial overview of where residuals are broadly positive or negative. These examples use the same event-centered residual field as the station-bias maps.


## Spatial Diagnostic Figures

Purpose: render spatial-correlation, PCA, clustering, and geology diagnostic figures from Step 4 outputs.

Outputs: spatial diagnostic figures plus compact diagnostic summary tables.

Render the spatial-correlation, PCA-summary, and geology-contrast figures from the Step 4 output tables. The package helper owns the metric filters and figure paths, and returns the compact diagnostic rows used to annotate the figures.


In [ ]:
# Render correlation, PCA, and geology diagnostic figures from the package-owned Step 4 suite.
# The result helper owns the metric-specific tables, configured figure paths, sidecars, and diagnostics.
spatial_diagnostic_result = spatial_outputs.write_diagnostic_figures(
    spatial_figure_settings,
    cfg=cfg,
)
display(spatial_diagnostic_result.preview_frame())
display(spatial_diagnostic_result.status_frame())


## Spatial Figure Provenance

Purpose: inspect compact provenance records for the Step 4 figure files.

Outputs: a preview table of figure metadata and sidecar paths.

Review the figure sidecar metadata written for this step without loading the full row CSV sidecars.


In [ ]:
display(spatial_figure_settings.sidecars.readiness_frame())
display(spatial_figure_settings.sidecars.status_frame())
